In [1]:
# --- Core Python ---
import os
import sys
from pathlib import Path
import h5py
import re

# Ensure notebook-local helper modules are importable
_NB_DIR = Path.cwd()
print(f"currently in {_NB_DIR}")
if (_NB_DIR / "resp_helper_functions.py").exists():
    print("helper functions exists in working directory, adding to sys.path")
    sys.path.insert(0, str(_NB_DIR))
else:
    # Fallback when notebook is launched from workspace root
    _NB_DIR = Path("notebooks/thomas_notebooks/resp_classification").resolve()
    if (_NB_DIR / "resp_helper_functions.py").exists():
        sys.path.insert(0, str(_NB_DIR))
    print("Helper functions didn't exist in working directory, resolving and adding to sys.path")

# --- Numerical & data analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Signal processing ---
from scipy.signal import butter, filtfilt, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Machine learning & metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Specialized neurophysiology tools ---
import neurokit2 as nk

# All the functions used to load, clean, and verify respiration and BORIS data
from resp_helper_functions import *

# --- Pandas display settings ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

currently in c:\Users\thoma\Code\ResearchCode\respiratory_pilot
Helper functions didn't exist in working directory, resolving and adding to sys.path
Libraries loaded successfully


In [2]:
# ======================================================
# Respiration + BORIS paths (VALENCE SETS ONLY)
# ======================================================

# Base paths for valence data
valence_h5_base = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\Resp_h5"
valence_boris_base = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\boris_csv_RI1 and RI2"

# --- Valence (RI1, RI2, BLRI) ---
# Using relative paths from base; adjust file names as needed
resp_paths_valence = {
    "RI1_3_6": os.path.join(valence_h5_base, "RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5"),
    "RI2_3_6": os.path.join(valence_h5_base, "RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5"),
    "RI1_4_7": os.path.join(valence_h5_base, "RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5"),
    "RI2_4_7": os.path.join(valence_h5_base, "RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5"),
    "RI1_2_3": os.path.join(valence_h5_base, "RI1_s2_3_p5_3_nRB3_20250622_104059_merged.h5"),
    "RI2_2_3": os.path.join(valence_h5_base, "RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5"),
    "RI1_4_8": os.path.join(valence_h5_base, "RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5"),
    "RI2_4_8": os.path.join(valence_h5_base, "RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5"),
    "RI1_1_1": os.path.join(valence_h5_base, "RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5"),
    "RI2_1_1": os.path.join(valence_h5_base, "RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5"),
    "RI1_1_2": os.path.join(valence_h5_base, "RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5"),
    "RI2_1_2": os.path.join(valence_h5_base, "RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5"),
    "RI1_2_4": os.path.join(valence_h5_base, "RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5"),
    "RI2_2_4": os.path.join(valence_h5_base, "RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5"),
    "RI1_3_5": os.path.join(valence_h5_base, "RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5"),
    "RI2_3_5": os.path.join(valence_h5_base, "RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5"),
    # Baseline recordings (pre-valence)
    "BLRI_1_1": os.path.join(valence_h5_base, "BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5"),
    "BLRI_1_2": os.path.join(valence_h5_base, "BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5"),
    "BLRI_2_3": os.path.join(valence_h5_base, "BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5"),
    "BLRI_2_4": os.path.join(valence_h5_base, "BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5"),
    "BLRI_3_6": os.path.join(valence_h5_base, "BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5"),
    "BLRI_4_7": os.path.join(valence_h5_base, "BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5"),
    "BLRI_4_8": os.path.join(valence_h5_base, "BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5"),
}

boris_paths_valence = {
    "RI1_3_6": os.path.join(valence_boris_base, "RI1_s3_6_p5_3_nRB3_HEEPS.csv"),
    "RI2_3_6": os.path.join(valence_boris_base, "RI2_s3_6_p_5_3_nRB3_2025062.csv"),
    "RI1_4_7": os.path.join(valence_boris_base, "RI1_s4_7_p5_2_nRB3_HEEPS.csv"),
    "RI2_4_7": os.path.join(valence_boris_base, "RI2_s4_7_p5_2_nRB3_HEEPS.csv"),
    "RI1_2_3": os.path.join(valence_boris_base, "RI1_s2_3_p5_3_nRB3_20250622_104059_ss.csv"),
    "RI2_2_3": os.path.join(valence_boris_base, "RI2_s2_3_p5_3_nRB3_20250622_1102116.1_ss.csv"),
    "RI1_4_8": os.path.join(valence_boris_base, "RI1_s4_8_p5_1_nRB3_HEEPS.csv"),
    "RI2_4_8": os.path.join(valence_boris_base, "RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv"),
    "RI1_1_1": os.path.join(valence_boris_base, "RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv"),
    "RI2_1_1": os.path.join(valence_boris_base, "RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv"),
    "RI1_1_2": os.path.join(valence_boris_base, "RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv"),
    "RI2_1_2": os.path.join(valence_boris_base, "RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv"),
    "RI1_2_4": os.path.join(valence_boris_base, "RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv"),
    "RI2_2_4": os.path.join(valence_boris_base, "RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv"),
    "RI2_3_5": os.path.join(valence_boris_base, "RI2_s3_5_p5_4_nRB3_20250621.csv"),
}

In [3]:
rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant"
}

In [4]:
# ======================================================
# Summary check (VALENCE ONLY)
# ======================================================

def summarize_paths(resp_dict, boris_dict, label):
    resp_set = set(resp_dict.keys())
    boris_set = set(boris_dict.keys())
    missing = resp_set - boris_set
    print(f"\n {label} summary")
    print(f"Resp files: {len(resp_set)} | BORIS files: {len(boris_set)} | No BORIS: {len(missing)}")
    if missing:
        print("Missing trials:", ", ".join(sorted(missing)))

# ======================================================
# Organized summaries by experiment type (VALENCE ONLY)
# ======================================================

# --- Split out valence subsets for clarity ---
resp_paths_interactions = {k: v for k, v in resp_paths_valence.items() if k.startswith(("RI1", "RI2"))}
resp_paths_blri = {k: v for k, v in resp_paths_valence.items() if k.startswith("BLRI")}

# Filter BORIS dicts by matching prefixes
boris_paths_valence = {k: v for k, v in boris_paths_valence.items() if k.startswith(("RI1", "RI2"))}

summarize_paths(resp_paths_valence, boris_paths_valence, "Valence (RI1/RI2)")

print(f"\nTotal respiration files (valence only): {len(resp_paths_valence)}")
print(f"Total BORIS files (valence only): {len(boris_paths_valence)}")

# Note: BLRI (pre-valence baseline) processing will be attempted but skipped if missing
print(f"\nPre-valence Baseline (BLRI) files available: {len(resp_paths_blri)}")


 Valence (RI1/RI2) summary
Resp files: 23 | BORIS files: 15 | No BORIS: 8
Missing trials: BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8, RI1_3_5

Total respiration files (valence only): 23
Total BORIS files (valence only): 15

Pre-valence Baseline (BLRI) files available: 7


In [5]:
def build_window_feature_matrix(
    resp_paths_interactions,
    boris_paths_valence,
    resp_paths_blri,
    rank_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=2.0,
    n_windows_per_session=100,
    min_breaths_per_window=2,
    random_state=42,
):
    all_rows = []

    # =========================================================
    # A. Interaction sessions (RI1 / RI2)
    # =========================================================
    for trial, h5_path in resp_paths_interactions.items():
        print(f"\n[Interaction] {trial}")

        signal, time, fs, meta = load_clean_resp_signal(
            h5_path,
            target_rate=target_srate
        )
        if signal is None:
            print("  ❌ Load failed")
            continue

        bm = fit_bm_session(signal, fs, data_type=data_type)
        if bm is None:
            continue

        print(f"  ✓ {len(bm.inhaleOnsets)} breaths detected in session")

        subj      = "_".join(trial.split("_")[1:3])
        condition = trial.split("_")[0]
        rank      = rank_map.get(subj, np.nan)

        # sample random windows from full session
        windows = sample_random_nonoverlapping_windows(
            time=time,
            window_dur=window_sec,
            n_windows=n_windows_per_session,
            seed=random_state,
            allow_partial_if_short=False,
        )

        if len(windows) == 0:
            print("  ⚠️ No usable full windows")
            continue

        window_rows = extract_features_from_windows(
            bm=bm,
            windows=windows,
            min_breaths_per_window=min_breaths_per_window,
        )

        print(f"  ✓ {len(window_rows)} usable windows")

        # optional BORIS summary per session
        n_bouts_by_behavior = {}
        total_bout_dur_by_behavior = {}

        if trial in boris_paths_valence:
            boris_df = load_clean_boris(boris_paths_valence[trial])
            if not boris_df.empty:
                for beh, grp in boris_df.groupby("Behavior"):
                    safe_beh = str(beh).replace(" ", "_")
                    n_bouts_by_behavior[f"n_bouts_{safe_beh}"] = len(grp)
                    total_bout_dur_by_behavior[f"total_dur_{safe_beh}"] = grp["Duration"].sum()

        for row in window_rows:
            row.update({
                "Trial": trial,
                "Subject": subj,
                "Condition": condition,
                "Rank": rank,
                "Type": "Interaction",
                "session_duration_sec": time[-1] - time[0],
                "n_breaths_total": len(bm.inhaleOnsets),
                "WindowSec": window_sec,
            })
            row.update(n_bouts_by_behavior)
            row.update(total_bout_dur_by_behavior)
            all_rows.append(row)

    # =========================================================
    # B. Baseline sessions (BLRI)
    # =========================================================
    for trial, h5_path in resp_paths_blri.items():
        print(f"\n[Baseline] {trial}")

        signal, time, fs, meta = load_clean_resp_signal(
            h5_path,
            target_rate=target_srate
        )
        if signal is None:
            print("  ❌ Load failed")
            continue

        bm = fit_bm_session(signal, fs, data_type=data_type)
        if bm is None:
            continue

        print(f"  ✓ {len(bm.inhaleOnsets)} breaths detected in session")

        subj      = "_".join(trial.split("_")[1:3])
        condition = trial.split("_")[0]
        rank      = rank_map.get(subj, np.nan)

        windows = sample_random_nonoverlapping_windows(
            time=time,
            window_dur=window_sec,
            n_windows=n_windows_per_session,
            seed=random_state,
            allow_partial_if_short=False,
        )

        if len(windows) == 0:
            print("  ⚠️ No usable full windows")
            continue

        window_rows = extract_features_from_windows(
            bm=bm,
            windows=windows,
            min_breaths_per_window=min_breaths_per_window,
        )

        print(f"  ✓ {len(window_rows)} usable windows")

        for row in window_rows:
            row.update({
                "Trial": trial,
                "Subject": subj,
                "Condition": condition,
                "Rank": rank,
                "Type": "Baseline",
                "session_duration_sec": time[-1] - time[0],
                "n_breaths_total": len(bm.inhaleOnsets),
                "WindowSec": window_sec,
            })
            all_rows.append(row)

    # =========================================================
    # C. Assemble
    # =========================================================
    if not all_rows:
        print("\n⚠️ No rows collected.")
        return pd.DataFrame()

    master_df = pd.DataFrame(all_rows).reset_index(drop=True)

    print(f"\n✅ Window feature matrix: {len(master_df)} rows × {len(master_df.columns)} cols")
    print(f"   Trials    : {master_df['Trial'].nunique()}")
    print(f"   Subjects  : {master_df['Subject'].nunique()}")
    print(f"   Type dist :\n{master_df['Type'].value_counts()}")

    return master_df

In [10]:
window_df = build_window_feature_matrix(
    resp_paths_interactions = resp_paths_interactions,
    boris_paths_valence     = boris_paths_valence,
    resp_paths_blri         = resp_paths_blri,
    rank_map                = rank_map,
    data_type               = "rodentAirflow",
    target_srate            = 400,
    window_sec              = 2.0,
    n_windows_per_session   = 100,
    min_breaths_per_window  = 2,
    random_state            = 42,
)


[Interaction] RI1_3_6
original resp len: 12065446
original fs: 20000.0
duration_sec: 603.2723
expected duration from len/fs: 603.2723
  ✓ 3902 breaths detected in session
  ✓ 97 usable windows

[Interaction] RI2_3_6
original resp len: 12200094
original fs: 20000.0
duration_sec: 610.0047
expected duration from len/fs: 610.0047
  ✓ 2506 breaths detected in session
  ✓ 87 usable windows

[Interaction] RI1_4_7
original resp len: 12583112
original fs: 20000.0
duration_sec: 629.1556
expected duration from len/fs: 629.1556
  ✓ 4118 breaths detected in session
  ✓ 100 usable windows

[Interaction] RI2_4_7
original resp len: 12129332
original fs: 20000.0
duration_sec: 606.4666
expected duration from len/fs: 606.4666
  ✓ 3592 breaths detected in session
  ✓ 100 usable windows

[Interaction] RI1_2_3
original resp len: 12058916
original fs: 20000.0
duration_sec: 602.9458
expected duration from len/fs: 602.9458
  ✓ 3654 breaths detected in session
  ✓ 93 usable windows

[Interaction] RI2_2_3
origi

In [11]:
window_df.head(23)

,WindowID,Start,Stop,Duration,n_breaths,breathing_rate_hz,mean_ibi_sec,cv_breathing_rate,mean_inhale_dur_sec,cv_inhale_dur,mean_exhale_dur_sec,cv_exhale_dur,ie_ratio,mean_peak_insp_flow,mean_peak_exp_flow,cv_peak_insp_flow,mean_inhale_vol,mean_exhale_vol,mean_tidal_vol,cv_tidal_vol,minute_ventilation,pct_breaths_with_inhale_pause,mean_inhale_pause_dur_sec,cv_inhale_pause_dur,inhale_pause_duty_cycle,pct_breaths_with_exhale_pause,mean_exhale_pause_dur_sec,cv_exhale_pause_dur,exhale_pause_duty_cycle,inhale_duty_cycle,exhale_duty_cycle,Trial,Subject,Condition,Rank,Type,session_duration_sec,n_breaths_total,WindowSec,n_bouts_anogenital_sniffing,n_bouts_body_sniffing,n_bouts_facial_sniffing,total_dur_anogenital_sniffing,total_dur_body_sniffing,total_dur_facial_sniffing
0,0,22.0,24.0,2.0,21,10.230179,0.097750,0.119386,0.040238,0.159159,0.056310,0.389133,0.714588,1097.355393,-1038.295346,0.314565,31851.248621,40006.925144,71858.173765,0.420068,735121.982251,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.411643,0.576057,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
1,1,30.0,32.0,2.0,19,9.230769,0.108333,0.129861,0.046711,0.153411,0.057500,0.198492,0.812357,1364.523730,-1053.449544,0.236839,44580.829707,41097.829927,85678.659635,0.364606,790879.935088,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.431174,0.530769,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
2,2,32.0,34.0,2.0,20,9.620253,0.103947,0.114765,0.043875,0.131764,0.055750,0.179988,0.786996,1149.622486,-1000.130706,0.215808,35579.550606,37648.419147,73227.969754,0.334772,704471.607757,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.422089,0.536329,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
3,3,34.0,36.0,2.0,18,9.497207,0.105294,0.102328,0.043750,0.201806,0.057361,0.185179,0.762712,1228.694587,-1056.541695,0.342465,39298.885158,40813.159328,80112.044486,0.501146,760840.645955,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.415503,0.544770,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
4,4,36.0,38.0,2.0,19,9.125475,0.109583,0.165082,0.048421,0.408333,0.059342,0.280398,0.815965,1190.780906,-1028.063969,0.326193,40437.061537,41064.216830,81501.278367,0.502602,743737.901446,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.441865,0.541525,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
5,5,38.0,40.0,2.0,18,9.418283,0.106176,0.103776,0.048056,0.123299,0.058472,0.404896,0.821853,1387.929677,-969.699373,0.247949,46352.905907,39194.096385,85547.002292,0.338973,805705.838759,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.452601,0.550708,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
6,6,40.0,42.0,2.0,17,9.103841,0.109844,0.309404,0.040735,0.146998,0.064265,0.465055,0.633867,921.663944,-953.869354,0.365055,27090.291987,41169.608259,68259.900246,0.497357,621427.256861,0.000000,0.0000,NaN,0.000000,0.058824,0.02750,0.000000,0.014727,0.370848,0.585056,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
7,7,42.0,44.0,2.0,19,9.350649,0.106944,0.236806,0.048158,0.230223,0.049737,0.208777,0.968254,1061.326426,-853.734000,0.415742,36067.039011,28556.203286,64623.242298,0.558816,604269.278629,0.052632,0.0525,0.0,0.025837,0.000000,0.00000,NaN,0.000000,0.450308,0.465072,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
8,8,56.0,58.0,2.0,17,8.951049,0.111719,0.191312,0.052647,0.134775,0.057941,0.405091,0.908629,1472.166504,-874.627756,0.291809,53119.481964,35383.404657,88502.886621,0.370864,792193.670451,0.000000,0.0000,NaN,0.000000,0.000000,0.00000,NaN,0.000000,0.471246,0.518634,RI1_3_6,3_6,RI1,Dominant,Interaction,603.27,3902,2.0,9.0,4.0,3.0,16.998,2.068,3.069
9,9,70.0,72.0,2.0,9,4.377565,0.228437,0.491500,0.116111,0.699171,0.068056,0.

In [12]:
window_df.to_pickle(r"C:\Users\thoma\Code\ResearchCode\respiratory_pilot\notebooks\thomas_notebooks\resp_classification\data\raw-breathmetrics-features-session-2sec")